# 실습 16 · 생성형 파운데이션 모델 — 신약 후보 분자 만들기 (De novo Design)
### AI 가 '읽는' 것을 넘어 '새로 그려내는' 단계

**왜 중요한가 (신약개발 AI 의 다음 단계)**
예제 10~15 는 이미 존재하는 분자를 **분석·예측**했습니다. 이번에는 반대로,
AI 가 세상에 없던 **새로운 분자를 스스로 설계(생성)** 합니다.
GPT 가 문장을 이어 쓰듯, **SMILES 를 학습한 생성모델**은 화학적으로 그럴듯한
새 분자를 한 글자씩 이어서 만들어냅니다. 이것이 신약개발의 후보물질 탐색
비용을 획기적으로 줄이는 **생성형 AI 신약설계**입니다.

**이 노트북에서 배우는 것**
1. 수백만 개 분자로 사전학습된 **생성모델**을 불러와 새 분자 **샘플링**
2. 만들어진 분자가 **화학적으로 유효한지** RDKit 으로 검증
3. **약물 유사성(Lipinski, QED)** 으로 좋은 후보만 걸러내는 **생성→필터 깔때기**
4. 특정 성질을 노리는 **조건부 생성(guided generation)** 개념

> 💡 런타임을 **T4 GPU** 로 바꾸면 훨씬 빠릅니다. (CPU 로도 동작합니다)


In [ ]:
!pip install transformers torch rdkit -q
import torch, numpy as np, pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, QED, Crippen
from rdkit import RDLogger; RDLogger.DisableLog("rdApp.*")   # 경고 숨김
device = "cuda" if torch.cuda.is_available() else "cpu"
print("준비 완료 ✅  (device:", device, ")")


## 1. ⭐ 분자 생성모델 불러오기 (SMILES 를 학습한 GPT)
`entropy/gpt2_zinc_87m` 은 **ZINC 데이터베이스의 수백만 분자**로 사전학습된
GPT-2 계열 생성모델입니다. 아무 입력 없이 실행하면, 모델이 학습한
'화학 언어' 로 **새로운 SMILES 를 처음부터 써 내려갑니다.**


In [ ]:
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast

model_name = "entropy/gpt2_zinc_87m"
tok = PreTrainedTokenizerFast.from_pretrained(model_name)
gen_model = GPT2LMHeadModel.from_pretrained(model_name).to(device).eval()
print("생성모델 로드 완료 ✅  (파라미터 약 87M, ZINC 사전학습)")


## 2. 새로운 분자 100개 샘플링하기
`do_sample=True` 는 매번 다른 결과를 뽑는 **창의적 생성** 모드입니다.
`temperature` 가 높을수록 더 과감(다양)하게, `top_k` 는 후보 글자 수를 제한합니다.


In [ ]:
N = 100
torch.manual_seed(42)
# bos 토큰에서 시작해 분자를 자유 생성
bos = torch.tensor([[tok.bos_token_id]] * N).to(device)
with torch.no_grad():
    out = gen_model.generate(
        bos, do_sample=True, max_length=64,
        temperature=1.0, top_k=30,
        pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
raw_smiles = tok.batch_decode(out, skip_special_tokens=True)
raw_smiles = [s.strip() for s in raw_smiles]
print("생성된 SMILES 예시 5개:")
for s in raw_smiles[:5]:
    print("  ", s)


## 3. 유효성 검증 — AI 가 만든 게 '진짜 분자' 일까?
생성모델은 가끔 문법이 틀린(=화학적으로 불가능한) SMILES 도 만듭니다.
RDKit 으로 **파싱이 되는 것만** 유효한 분자로 인정합니다.
유효율(validity)은 생성모델의 품질 지표입니다.


In [ ]:
valid = []
for s in raw_smiles:
    m = Chem.MolFromSmiles(s)
    if m is not None:
        valid.append(Chem.MolToSmiles(m))   # 표준화(canonical)
valid = list(dict.fromkeys(valid))          # 중복 제거
print(f"생성 {len(raw_smiles)}개 중 유효 & 고유 분자: {len(valid)}개")
print(f"유효율(validity): {len(valid)/len(raw_smiles)*100:.0f}%")

# 학습 데이터에 없던 '새로운' 분자인지도 확인 가능(여기선 개념만)
print("→ 이 분자들은 대부분 데이터베이스에 없던 새 구조입니다 (de novo)")


## 4. 약물 유사성 필터 — 좋은 후보만 남기기 (생성→필터 깔때기)
분자를 많이 만드는 것보다 **쓸 만한 것을 고르는 것**이 중요합니다.
예제 12(가상 스크리닝)처럼, 약물다움 규칙으로 깔때기를 통과시킵니다.
- **Lipinski Rule of 5**: 경구 약물이 되기 좋은 조건
- **QED**: 0~1 사이 '약물다움' 종합 점수 (높을수록 좋음)


In [ ]:
def profile(smi):
    m = Chem.MolFromSmiles(smi)
    mw   = Descriptors.MolWt(m)
    logp = Crippen.MolLogP(m)
    hbd  = Descriptors.NumHDonors(m)
    hba  = Descriptors.NumHAcceptors(m)
    qed  = QED.qed(m)
    lipinski = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)
    return dict(SMILES=smi, MW=round(mw,1), logP=round(logp,2),
                HBD=hbd, HBA=hba, QED=round(qed,3), Lipinski=lipinski)

df = pd.DataFrame([profile(s) for s in valid])
# 깔때기: Lipinski 통과 + QED 상위
hits = df[df["Lipinski"]].sort_values("QED", ascending=False).reset_index(drop=True)
print(f"Lipinski 통과: {df['Lipinski'].sum()} / {len(df)}")
print("QED 기준 상위 후보 5개:")
hits.head(5)


## 5. 상위 후보 분자 그려보기
AI 가 설계한 신약 후보들을 눈으로 확인합니다.


In [ ]:
top = hits.head(6)
mols = [Chem.MolFromSmiles(s) for s in top["SMILES"]]
legends = [f"QED={q}" for q in top["QED"]]
Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(240,200), legends=legends)


## 6. (심화) 조건부 생성 개념 — '원하는 성질' 을 노리기
현업에서는 그냥 아무거나 만들지 않고 **목표 성질을 향해** 생성을 유도합니다.
간단한 버전으로, 많이 생성한 뒤 **원하는 조건으로 필터**하면 조건부 생성과
비슷한 효과를 냅니다. (진짜 조건부 생성은 REINVENT·강화학습·조건토큰을 사용)


In [ ]:
# 예: '작고(MW<350) 지용성 적당한(1<logP<3)' 후보만 원한다면
target = df[(df.MW < 350) & (df.logP.between(1,3)) & (df.Lipinski)]
target = target.sort_values("QED", ascending=False)
print(f"목표 조건을 만족하는 후보: {len(target)}개")
target.head(5)


## 7. 생성형 신약설계 지형도 (참고)
| 방식 | 대표 도구 | 특징 |
|---|---|---|
| **언어모델 생성** | GPT/ZINC, ChemGPT, MolGPT | SMILES/SELFIES 를 이어 씀 (이 예제) |
| **강화학습 최적화** | REINVENT | 목표 점수(활성·ADMET)로 생성 유도 |
| **확산모델** | 3D 분자·결합구조 생성 | 단백질 포켓에 맞춰 설계 |
| **VAE/GraphGen** | JT-VAE, GraphAF | 잠재공간 탐색으로 신규 구조 |

**표준 워크플로우**: 대량 생성 → 유효성 검증 → ADMET·약물다움 필터 →
도킹(예제 14)·활성예측(예제 11)으로 검증 → 유망 후보 합성/실험


## 정리 & 현장 응용
- **생성형 AI 신약설계**: 분자를 '분석' 하는 것을 넘어 '새로 만든다'
- **생성 → 검증 → 필터 깔때기**: 많이 만들고, 유효성·약물다움으로 좁힌다
- 이후 예제 11(활성)·12(스크리닝)·14(도킹)과 연결하면 **후보발굴 파이프라인 완성**
- 확장: REINVENT 로 목표 성질 최적화, 확산모델로 단백질 포켓 맞춤 설계
- **면접 한 문장**: "SMILES 를 학습한 생성모델로 신규 후보를 대량 설계하고,
  RDKit 유효성·약물다움 필터와 도킹으로 좁히는 생성형 신약설계 파이프라인을
  경험했습니다."
